# 03. Evaluating & Auto-Improving Agent Responses (LLM-as-Judge)

This notebook (**AI-103 -> Section 03**) builds a small three-pass pipeline around the CloudXeus RAG support agent: (1) get a **draft** grounded answer from the agent, (2) have a second LLM call act as a **judge**, critiquing the draft against a completeness checklist and replying only `COMPLETE` or `MISSING`, and (3) **conditionally regenerate** a more thorough answer only if the judge found it incomplete. It's a hand-rolled evaluation gate you'd otherwise get more formally from Azure AI Foundry's built-in evaluators. The companion no-code lab `04. Lab - Adding conditions to the workflow/workflow.yml` implements a comparable conditional-branching idea declaratively, in the Foundry portal's Workflow feature.

**Difficulty: Advanced**


## Prerequisites

**Python packages** (already in this repo's root `requirements.txt`):
- `azure-ai-projects`, `azure-identity`, `python-dotenv`

**Azure resources**
- An Azure AI Foundry project with a **RAG-grounded** Agent published (connected to a knowledge source such as a vector store or Azure AI Search index -- conceptually the same pattern as `11_azure_ai_foundry/07_knowledge_rag/` in this repo). It's built via the same companion lab referenced in the previous notebook, `02. Lab - Creating the CloudXeus Agents/`.
- Entra ID auth via `az login`, with an appropriate RBAC role (e.g. Azure AI Developer) on the project.

**Environment variables** -- add/reuse in the root `.env`:
```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_AGENT_NAME=cloudxeus-support-rag-agent
AZURE_AI_MODEL_DEPLOYMENT=<your-deployment-name>
```
> The original script hardcodes `model="gpt-5.4"` -- a course-specific deployment alias, not a real public model name. Set `AZURE_AI_MODEL_DEPLOYMENT` to whatever deployment actually exists in your own project.


## What You'll Learn
- The **LLM-as-judge** pattern: one model call critiques another's output against a rubric
- Why the critique call deliberately omits `agent_reference` (judge only the given text, don't let the judge re-query the knowledge base itself)
- Prompting a judge to avoid penalizing an answer for gaps it explicitly disclosed
- Conditional regeneration driven by a parsed model output (`"MISSING" in critique.upper()`)
- The cost/latency trade-off of multi-pass LLM pipelines


### 1. Imports, configuration, and the project client

Same `AIProjectClient` + `get_openai_client()` bridge as the previous notebook. `MODEL_DEPLOYMENT` replaces the course's hardcoded `"gpt-5.4"` placeholder.

**Exam tip:** Which RBAC role an identity holds on a Foundry project governs whether it can *call* agents (e.g. Azure AI Developer / Cognitive Services User) versus *manage* them (create, publish, delete). Assigning the narrowest role that still lets a given script or service principal do its job is the kind of least-privilege governance question AI-102 covers for generative AI solutions.


In [ ]:
import os

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "cloudxeus-support-rag-agent")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT", "gpt-5.4")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai_client = project.get_openai_client()


### 2. Define the reusable agent reference and the scenario question

`AGENT_REF` is built once and reused across every call that needs to hit the grounded agent -- small production hygiene that avoids retyping (and risking a typo in) the same dict repeatedly. `user_question` is the customer's message for this scenario: a refund-eligibility question that should be answerable from CloudXeus's own policy documents.


In [ ]:
AGENT_REF = {"name": AGENT_NAME, "type": "agent_reference"}

user_question = (
    "I'm on the Pro plan and want a refund for my last subscription charge. "
    "Am I eligible, and is there anything I should know before I request it?"
)


### 3. First pass -- draft response from the grounded agent

`model=MODEL_DEPLOYMENT` is still required by the API's schema, but when `extra_body` carries `agent_reference`, the call is routed to the named Agent -- whose own configured model, instructions, and connected knowledge source (the RAG piece) actually determine the answer.

**Exam tip:** Foundry Agents can be connected to a knowledge source (a vector store or an Azure AI Search index) via file search / RAG, grounding answers in your own documents rather than the model's parametric memory alone -- this materially reduces hallucination on policy-specific questions like refund eligibility, at the cost of needing those documents indexed and kept current.


In [ ]:
# First pass: ask the grounded agent
draft_response = openai_client.responses.create(
    model=MODEL_DEPLOYMENT,
    input=user_question,
    extra_body={"agent_reference": AGENT_REF},
)

answer = draft_response.output_text

print("Draft response")
print(answer)


### 4. Second pass -- an LLM judge critiques the draft for completeness

Notice this call carries **no** `extra_body`/`agent_reference` -- it's a plain model call over the question and the draft answer text only, deliberately *not* re-querying the knowledge base. The judge should evaluate what the draft actually said, not go looking for more information itself. The prompt spells out a five-point completeness checklist and explicitly instructs the judge not to penalize the draft for a gap it already disclosed as "not available in the knowledge base" -- this reduces false negatives where the judge would otherwise criticize the agent for correctly declining to answer something it doesn't know. The reply is constrained to exactly `COMPLETE` or `MISSING` so the result is trivial to parse programmatically.

**Exam tip:** This hand-rolled critique loop is a simplified version of what Azure AI Foundry's built-in **evaluators** (the `azure-ai-evaluation` SDK / the Foundry portal's Evaluation feature) do more rigorously -- e.g. built-in Completeness, Groundedness, and Relevance evaluators score generated answers against consistent rubrics at scale, rather than relying on one hand-written judge prompt per app.


In [ ]:
# Second pass: check whether the answer is complete
critique_response = openai_client.responses.create(
    model=MODEL_DEPLOYMENT,
    input=(
        "You are reviewing a customer support answer for completeness.\n\n"
        "A complete refund answer should address:\n"
        "1. Whether the customer appears eligible for a refund.\n"
        "2. The refund window, if one applies.\n"
        "3. How the refund window is measured.\n"
        "4. Whether refunds are available after the normal refund window.\n"
        "5. Whether any requested details are not available in the knowledge base.\n\n"
        "Do not judge the answer based on information that is not present. "
        "If the answer clearly says that a detail is not available in the knowledge base, "
        "treat that as acceptable.\n\n"
        f"Customer question:\n{user_question}\n\n"
        f"Support answer:\n{answer}\n\n"
        "Reply with only COMPLETE or MISSING."
    ),
)

critique = critique_response.output_text

print("Critique")
print(critique)


### 5. Third pass -- regenerate only if the judge said it's incomplete

The regeneration call only fires if `"MISSING"` appears in the judge's reply, so this third (most expensive) pass is skipped entirely on a complete draft. When it does fire, it goes back through the grounded agent (`extra_body=AGENT_REF` again) with a more explicit instruction covering all five checklist points, making a first-try success on the retry more likely.

**Exam tip:** Worst case, this pipeline makes three separate LLM calls per user question -- a real design trade-off between answer quality/completeness and cost + latency. AI-102 frames exactly this kind of quality-gate-vs-cost trade-off as a governance and monitoring concern for production generative AI applications, and it's also precisely what the companion no-code lab (`04. Lab - Adding conditions to the workflow/workflow.yml`) expresses declaratively as a workflow condition instead of custom Python.


In [ ]:
# Third pass: regenerate only if the answer is incomplete
if "MISSING" in critique.upper():
    print("Regenerated response")

    improved_response = openai_client.responses.create(
        model=MODEL_DEPLOYMENT,
        input=(
            "Answer the customer question completely using the available knowledge base. "
            "Cover refund eligibility, the refund window, how the window is measured, "
            "what happens after the refund window, and whether any requested details "
            "are not available in the knowledge base. "
            "Do not invent policy details.\n\n"
            f"Customer question:\n{user_question}"
        ),
        extra_body={"agent_reference": AGENT_REF},
    )

    print(improved_response.output_text)


## Summary

You ran a draft -> critique -> conditional-regenerate pipeline: the RAG agent produced a first answer, a separate plain model call judged it against a five-point completeness checklist (`COMPLETE`/`MISSING`), and a third, agent-grounded call only ran if the judge flagged the draft as incomplete. This is a lightweight, code-first analogue to Azure AI Foundry's built-in evaluation and workflow-condition features.


## Try It Yourself
1. **Easy:** Change `user_question` to a different policy scenario (e.g. a cancellation question instead of a refund).
2. **Medium:** Instead of only printing each pass, append `{"draft": ..., "critique": ..., "improved": ...}` to a list of dicts so you can review or log the full pipeline output later.
3. **Advanced:** Replace the hand-rolled critique prompt with Azure AI Foundry's `azure-ai-evaluation` Completeness evaluator (or a custom evaluator you define) and compare its verdict against this notebook's judge prompt on the same draft answer.
